In [4]:
'''
Sentiment scoring (build multiple, then compare)

Lexicon-based: VADER (general-purpose, fast baseline), Loughran-McDonald (finance-specific, handles words like "liability" or "tax" that VADER misreads as negative)
Bag-of-words + ML: TF-IDF/CountVectorizer → feed into logistic regression as your simplest learned model
Transformer-based: FinBERT as your top-line model
'''

'\nSentiment scoring (build multiple, then compare)\n\nLexicon-based: VADER (general-purpose, fast baseline), Loughran-McDonald (finance-specific, handles words like "liability" or "tax" that VADER misreads as negative)\nBag-of-words + ML: TF-IDF/CountVectorizer → feed into logistic regression as your simplest learned model\nTransformer-based: FinBERT as your top-line model\n'

In [5]:
'''
Score	Description
Positive (pos)	Represents the proportion of text that conveys positive sentiment.
Negative (neg)	Represents the proportion of text that conveys negative sentiment.
Neutral (neu)	Represents the proportion of text that is emotionally neutral.
Compound	A normalized score between -1 and +1 that indicates the overall sentiment of the text.
'''

'\nScore\tDescription\nPositive (pos)\tRepresents the proportion of text that conveys positive sentiment.\nNegative (neg)\tRepresents the proportion of text that conveys negative sentiment.\nNeutral (neu)\tRepresents the proportion of text that is emotionally neutral.\nCompound\tA normalized score between -1 and +1 that indicates the overall sentiment of the text.\n'

# Load earnings call transcripts and score sentiment with VADER and Loughran-McDonald

This notebook loads every transcript under the `Transcripts` folder, extracts each prepared remarks section, and compares two lexicon-based sentiment methods:
- VADER: fast general-purpose baseline
- Loughran-McDonald: finance-focused lexicon for words like `liability`, `tax`, and `risk`

The goal is to build a reproducible sentiment table before moving to ML or transformer models.

In [6]:
from pathlib import Path

import pandas as pd

from src.lexicon_sentiment import (
    load_transcripts,
    score_loughran_mcdonald,
    score_vader,
)

# Use the project root so the notebook works from the repo root or from the src folder.
project_root = Path.cwd().resolve()
if (project_root / 'Transcripts').exists():
    transcripts_dir = project_root / 'Transcripts'
else:
    transcripts_dir = project_root.parent / 'Transcripts'

print(f'Loading transcripts from: {transcripts_dir}')
transcripts = load_transcripts(transcripts_dir)
print(f'Loaded {len(transcripts)} transcript rows')
transcripts.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
transcripts['vader_score'] = transcripts['prepared_remarks'].apply(score_vader)
transcripts['lm_score'] = transcripts['prepared_remarks'].apply(score_loughran_mcdonald)

transcripts['vader_compound'] = transcripts['vader_score'].apply(lambda x: x.get('compound', 0.0))
transcripts['vader_pos'] = transcripts['vader_score'].apply(lambda x: x.get('pos', 0.0))
transcripts['vader_neg'] = transcripts['vader_score'].apply(lambda x: x.get('neg', 0.0))
transcripts['lm_compound'] = transcripts['lm_score'].apply(lambda x: x.get('compound', 0.0))
transcripts['lm_positive'] = transcripts['lm_score'].apply(lambda x: x.get('positive', 0.0))
transcripts['lm_negative'] = transcripts['lm_score'].apply(lambda x: x.get('negative', 0.0))

transcripts[['ticker', 'filename', 'vader_compound', 'lm_compound']].head(10)

,ticker,filename,vader_compound,lm_compound
0,AAPL,2016-Apr-26-AAPL.txt,0.0000,0.000000
1,AAPL,2016-Jan-26-AAPL.txt,0.9998,0.679012
2,AAPL,2016-Jul-26-AAPL.txt,0.9998,0.787234
3,AAPL,2016-Oct-25-AAPL.txt,0.9999,0.636364
4,AAPL,2017-Aug-01-AAPL.txt,0.9998,0.947368
5,AAPL,2017-Jan-31-AAPL.txt,0.9997,0.914286
6,AAPL,2017-May-02-AAPL.txt,0.9998,0.902439
7,AAPL,2017-Nov-02-AAPL.txt,0.9998,0.971429
8,AAPL,2018-Feb-01-AAPL.txt,0.9998,0.835294
9,AAPL,2018-Jul-31-AAPL.txt,0.9999,0.830986


In [ ]:
summary = (
    transcripts.groupby('ticker')
    .agg(
        vader_mean=('vader_compound', 'mean'),
        lm_mean=('lm_compound', 'mean'),
        transcripts_count=('filename', 'count')
    )
    .sort_values('vader_mean', ascending=False)
)

summary.head(10)

,vader_mean,lm_mean,transcripts_count
ticker,,,
AAPL,0.894595,0.710703,19
ASML,0.137868,0.052632,19
AMD,0.000000,0.000000,19
AMZN,0.000000,0.000000,19
CSCO,0.000000,0.000000,19
GOOGL,0.000000,0.000000,19
INTC,0.000000,0.000000,19
MSFT,0.000000,0.000000,19
MU,0.000000,0.000000,17


In [ ]:
# Save the scored output for downstream modeling or analysis
output_path = project_root / 'transcript_lexicon_scores.csv'
transcripts.to_csv(output_path, index=False)
print(f'Saved scored results to: {output_path}')

# Quick check of the result file
pd.read_csv(output_path).head()

Saved scored results to: /Users/nok/NLP-sentiment-signal-on-earnings-news/transcript_lexicon_scores.csv


,ticker,filename,transcript_path,prepared_remarks,raw_text,vader_score,lm_score,vader_compound,vader_pos,vader_neg,lm_compound,lm_positive,lm_negative
0,AAPL,2016-Apr-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Transcr...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...","{'positive': 0.0, 'negative': 0.0, 'neutral': ...",0.0000,0.000,0.000,0.000000,0.0,0.0
1,AAPL,2016-Jan-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.018, 'neu': 0.839, 'pos': 0.144, 'co...","{'positive': 68.0, 'negative': 13.0, 'neutral'...",0.9998,0.144,0.018,0.679012,68.0,13.0
2,AAPL,2016-Jul-26-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.012, 'neu': 0.854, 'pos': 0.135, 'co...","{'positive': 42.0, 'negative': 5.0, 'neutral':...",0.9998,0.135,0.012,0.787234,42.0,5.0
3,AAPL,2016-Oct-25-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.014, 'neu': 0.821, 'pos': 0.165, 'co...","{'positive': 45.0, 'negative': 10.0, 'neutral'...",0.9999,0.165,0.014,0.636364,45.0,10.0
4,AAPL,2017-Aug-01-AAPL.txt,/Users/nok/NLP-sentiment-signal-on-earnings-ne...,==============================================...,\n\nThomson Reuters StreetEvents Event Brief\n...,"{'neg': 0.018, 'neu': 0.839, 'pos': 0.144, 'co...","{'positive': 37.0, 'negative': 1.0, 'neutral':...",0.9998,0.144,0.018,0.947368,37.0,1.0


# Bag-of-words baseline with TF-IDF + logistic regression

This is the simplest learned model in the pipeline: convert each transcript into TF-IDF features and train a logistic regression classifier on a binary sentiment label derived from the VADER compound score.

The goal here is to create a baseline that is fast to train and easy to compare against the lexicon methods.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Create a binary label from the VADER compound score.
# A score > 0.05 is treated as positive, < -0.05 as negative, and the rest as neutral.
# We keep a simple 2-class setup for the first baseline.
transcripts['sentiment_label'] = transcripts['vader_compound'].apply(
    lambda x: 1 if x > 0.05 else 0
)

# Optional: remove neutral observations so the classifier is a clean positive/negative baseline.
# If you want to keep all rows, comment out the next line.
# transcripts = transcripts[transcripts['vader_compound'].abs() > 0.05].copy()

X = transcripts['prepared_remarks'].fillna('')
y = transcripts['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train)

preds = clf.predict(X_test_tfidf)

print('Accuracy:', round(accuracy_score(y_test, preds), 4))
print('\nClassification report:\n', classification_report(y_test, preds, digits=4))
print('\nConfusion matrix:\n', confusion_matrix(y_test, preds))

# Useful baseline metrics for later comparison
baseline_metrics = {
    'model': 'tfidf_logistic_regression',
    'accuracy': round(accuracy_score(y_test, preds), 4),
    'n_train': len(X_train),
    'n_test': len(X_test),
}
baseline_metrics

Accuracy: 0.9737

Classification report:
               precision    recall  f1-score   support

           0     0.9714    1.0000    0.9855        34
           1     1.0000    0.7500    0.8571         4

    accuracy                         0.9737        38
   macro avg     0.9857    0.8750    0.9213        38
weighted avg     0.9744    0.9737    0.9720        38


Confusion matrix:
 [[34  0]
 [ 1  3]]


{'model': 'tfidf_logistic_regression',
 'accuracy': 0.9737,
 'n_train': 150,
 'n_test': 38}

: 

# Transformer-based FinBERT baseline

This is the top-line model in the project: a finance-specific transformer that can capture context beyond single-word sentiment lexicons.

Use the `ProsusAI/finbert` model, which is a BERT model fine-tuned for financial sentiment classification.

Typical workflow:
1. Install the Hugging Face libraries.
2. Load the transcript text.
3. Run the model on each prepared remarks section.
4. Collect `positive`, `negative`, and `neutral` probabilities.
5. Compare FinBERT outputs against VADER and Loughran-McDonald.

This cell is intentionally documented as a setup template; the training run is heavy and usually needs a dedicated environment or GPU.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

device = "cpu"  # switch to "mps" once this works, as a quick A/B test

MODEL_NAME = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

LABELS = model.config.id2label  # {0: 'positive', 1: 'negative', 2: 'neutral'} for FinBERT

def chunk_text(text, max_tokens=510, stride=50):
    """Split long text into overlapping token chunks that fit BERT's 512-token limit."""
    ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(ids), max_tokens - stride):
        chunk_ids = ids[i:i + max_tokens]
        chunks.append(tokenizer.decode(chunk_ids))
        if i + max_tokens >= len(ids):
            break
    return chunks if chunks else [text]

@torch.no_grad()
def score_batch(texts, batch_size=16):
    """Score a list of (short) texts in batches. Returns list of {label: prob} dicts."""
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Scoring"):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=512
        ).to(device)
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        for p in probs:
            results.append({LABELS[j]: p[j].item() for j in range(len(LABELS))})
    return results

def score_long_document(text):
    """For a full transcript: chunk it, score each chunk, average the probabilities."""
    chunks = chunk_text(text)
    chunk_scores = score_batch(chunks, batch_size=8)
    avg = {label: sum(s[label] for s in chunk_scores) / len(chunk_scores) for label in LABELS.values()}
    return avg

/Users/nok/opt/anaconda3/envs/test_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/nok/opt/anaconda3/envs/test_env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
import sys
print(sys.executable)

/opt/homebrew/opt/python@3.11/bin/python3.11
